In [ ]:
import json, subprocess, sys, time, hashlib, shutil, zipfile, resource
from pathlib import Path
import torch
input_root=Path('/kaggle/input')
print('Mounted Kaggle inputs:',[p.name for p in input_root.iterdir()])
manifest_candidates=list(input_root.rglob('shared-html-reviewed-pages.jsonl'))
if not manifest_candidates: raise FileNotFoundError('shared-html-reviewed-pages.jsonl not mounted under /kaggle/input')
input_root=manifest_candidates[0].parent
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Kaggle GPU was not allocated')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','docling==2.129.0'],check=True)
from importlib.metadata import version
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import LayoutObjectDetectionOptions, PdfPipelineOptions, TableFormerMode, TableStructureOptions, TesseractCliOcrOptions
from docling.datamodel.stage_model_specs import ObjectDetectionModelSpec
from docling.document_converter import DocumentConverter, ImageFormatOption
manifest_path=input_root/'shared-html-reviewed-pages.jsonl'
manifest=[json.loads(line) for line in manifest_path.read_text().splitlines() if line.strip()]
output_root=Path('/kaggle/working/raw_pages'); output_root.mkdir(parents=True,exist_ok=True)
options=PdfPipelineOptions(accelerator_options=AcceleratorOptions(device=AcceleratorDevice.CUDA,num_threads=2),do_ocr=True,do_table_structure=True,layout_options=LayoutObjectDetectionOptions(model_spec=ObjectDetectionModelSpec(name='layout_heron',repo_id='docling-project/docling-layout-heron')),table_structure_options=TableStructureOptions(mode=TableFormerMode.ACCURATE,do_cell_matching=True),ocr_options=TesseractCliOcrOptions(lang=['vie','eng'],psm=6))
converter=DocumentConverter(format_options={InputFormat.IMAGE:ImageFormatOption(pipeline_options=options)})
started=time.perf_counter(); pages=[]
for item in manifest:
    page_started=time.perf_counter(); page_id=item['reference_id']; image=input_root/item['image_file']
    try:
        converted=converter.convert(image)
        (output_root/f'{page_id}.json').write_text(json.dumps(converted.document.export_to_dict(),ensure_ascii=False)+'\n')
        pages.append({'reference_id':page_id,'status':str(converted.status),'seconds':round(time.perf_counter()-page_started,3)})
        print(page_id,converted.status,flush=True)
    except Exception as exc:
        pages.append({'reference_id':page_id,'status':'failed','error':repr(exc),'seconds':round(time.perf_counter()-page_started,3)})
        print(page_id,'FAILED',repr(exc),flush=True)
metadata={'dataset':'letrongnhan/vietnamese-enterprise-e2e-hard-scans','input_manifest_sha256':hashlib.sha256(manifest_path.read_bytes()).hexdigest(),'input_pages':len(manifest),'device':torch.cuda.get_device_name(0),'cuda_available':True,'docling_version':version('docling'),'ocr':'Tesseract vie+eng PSM 6','layout_model':'docling-project/docling-layout-heron','table_structure':'Docling TableFormer accurate + OCR cell matching','inference_wall_seconds':round(time.perf_counter()-started,3),'successful_pages':sum(p['status']!='failed' for p in pages),'failed_pages':sum(p['status']=='failed' for p in pages),'pages':pages}
Path('/kaggle/working/kaggle-inference-metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2)+'\n')
shutil.make_archive('/kaggle/working/raw_pages','zip',root_dir=output_root)
print(json.dumps({k:metadata[k] for k in ('device','input_pages','successful_pages','failed_pages','inference_wall_seconds')}),flush=True)
